# RGB-only Agentic Memory：从视频到推理与导航

本教程完整演示：

```text
RGB -> LingBot depth + c2w -> stable local submap -> spatial memory
RGB -> VLM semantics -> Pi -> Scene Graph -> Knowledge Memory
RGB+ Knowledge Memory + Scene Graph -> Native Reasoner -> Navigation Action
```

运行时只依赖 RGB。GT、LiDAR 和 Isaac Sim 只用于离线评估。

## 1. 使用项目 Python 3.12 kernel

请在 VS Code notebook kernel 菜单中选择 **AgenticMemoryNav Python 3.12**。项目使用 `StrEnum`，Python 3.10 kernel 无法执行核心代码。

In [1]:
import sys
from pathlib import Path

if sys.version_info < (3, 11):
    raise RuntimeError('Select the AgenticMemoryNav Python 3.12 kernel, then rerun.')

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

import numpy as np
print('Python:', sys.version.split()[0])
print('Interpreter:', sys.executable)
print('Project root:', project_root)

Python: 3.12.3
Interpreter: /home/snt/projects/AgenticMemoryNav/.venv/bin/python
Project root: /home/snt/projects/AgenticMemoryNav


## 2. LingBot depth+c2w 如何形成稳定 local submap

LingBot 的 depth head、camera head 和内参使每个 RGB frame 可以反投影为局部点云。机器人移动时，视野中心会变化，因此不能使用点云质心静止作为稳定条件。

当前 gate 使用相邻点云的对称最近邻残差：

$$r(P_t,P_{t+1})=\frac{1}{2}(d(P_t,P_{t+1})+d(P_{t+1},P_t))$$

窗口中的最大残差低于阈值时，stable RGB-only submap 会写入 `MemoryType.SPATIAL`，并保存 residual、confidence、frame IDs 和 provenance。

In [2]:
from agentic_memory_nav.common.types import MappingUpdate, Pose3D
from agentic_memory_nav.mapping.local_submap import LocalSubmapBuilder

def mapping_update(index, offset_x):
    cloud = np.array([
        [offset_x, 0.0, 1.0],
        [offset_x + 0.02, 0.0, 1.0],
        [offset_x, 0.02, 1.0],
    ], dtype=np.float32)
    return MappingUpdate(
        frame_id=f'frame_{index:04d}', timestamp=float(index),
        camera_pose=Pose3D(position=(offset_x, 0.0, 0.0)),
        depth=np.ones((2, 2), dtype=np.float32),
        confidence=np.ones((2, 2), dtype=np.float32),
        local_pointcloud=cloud, global_pointcloud=cloud,
        is_keyframe=True, map_version=index + 1,
    )

builder = LocalSubmapBuilder(window_frames=3, frame_stride=1, stability_threshold_m=0.10)
submap = None
for index, offset_x in enumerate((0.00, 0.03, 0.06)):
    submap = builder.add(mapping_update(index, offset_x))

assert submap is not None
assert submap.stable
print('stable:', submap.stable)
print('frames:', submap.frame_ids)
print('overlap residual (m):', round(submap.geometric_residual_m, 4))
print('point count:', len(submap.points))

stable: True
frames: ['frame_0000', 'frame_0001', 'frame_0002']
overlap residual (m): 0.0221
point count: 9


In [ ]:
unstable_builder = LocalSubmapBuilder(window_frames=3, frame_stride=1, stability_threshold_m=0.10)
unstable_builder.add(mapping_update(0, 0.00))
unstable_builder.add(mapping_update(1, 0.03))
unstable = unstable_builder.add(mapping_update(2, 2.00))

assert unstable is not None
assert not unstable.stable
print('stable:', unstable.stable)
print('overlap residual (m):', round(unstable.geometric_residual_m, 4))
print('decision: do not commit this geometry window')

## 3. VLM semantics + depth geometry 如何形成 $P_i$

VLM 给出类别、属性和 2D bbox。bbox fallback 或后续 SAM mask 从 depth 中选取像素，再通过 intrinsics 和 c2w 反投影为对象点云 $P_i$。

```text
VLM bbox -> instance mask -> depth pixels -> world points -> Pi NPZ artifact
```

未来可替换为开放式点云实例分割模型；$P_i$、graph 和 memory 的接口不需要重写。

In [ ]:
from agentic_memory_nav.common.types import CameraIntrinsics, FrameObservation, ObjectObservation
from agentic_memory_nav.geometry.pointcloud_store import PointCloudStore
from agentic_memory_nav.perception.instance_segmentation import BoundingBoxSegmenter, InstanceGeometryEnricher

rgb = np.zeros((48, 64, 3), dtype=np.uint8)
depth = np.full((48, 64), 2.0, dtype=np.float32)
frame = FrameObservation(
    frame_id='pi_frame', timestamp=0.0, rgb=rgb, depth=depth,
    camera_intrinsics=CameraIntrinsics(60.0, 60.0, 32.0, 24.0, 64, 48),
    camera_pose=Pose3D(),
)
mapping = mapping_update(0, 0.0)
mapping.depth = depth
mapping.confidence = np.ones_like(depth)
cube = ObjectObservation(
    observation_id='obs_red_cube', category='cube', attributes={'color': 'red'},
    bbox_2d=(20, 12, 44, 36), center_3d=(0.0, 0.0, 0.0),
    dimensions_3d=(0.0, 0.0, 0.0), confidence=0.9,
    timestamp=0.0, frame_id=frame.frame_id,
)

pi_store = PointCloudStore(Path('/tmp/agentic_memory_nav_workshop_v3_pi'))
cube = InstanceGeometryEnricher(BoundingBoxSegmenter(), pi_store).enrich(frame, mapping, [cube])[0]

assert cube.geometry is not None
print('Pi artifact:', cube.geometry.artifact_path)
print('Pi point count:', cube.geometry.point_count)
print('Pi centroid:', cube.geometry.centroid_3d)

## 4. Scene Graph -> Knowledge Memory

对象 $P_i$ 进入 Scene Graph 后成为 object node。room node 与 object node 通过 `inside` 等 relation edge 连接。`KnowledgeMemory` 再把这些节点和有向边 materialize 为 SQLite memory graph facts。

这样推理不是直接依赖一次 VLM 回答，而是依赖带 provenance 的历史 graph evidence。

In [ ]:
from agentic_memory_nav.memory.knowledge_memory import KnowledgeMemory
from agentic_memory_nav.memory.sqlite_store import SQLiteMemory
from agentic_memory_nav.scene_graph.graph import SceneGraph
from agentic_memory_nav.scene_graph.updater import SceneGraphUpdater

# room 与 cube 位置分开，association 不会把两个不同类别合并成同一个 node。
room = ObjectObservation(
    observation_id='obs_kitchen', category='kitchen', attributes={'kind': 'room'},
    bbox_2d=(0, 0, 64, 48), center_3d=(3.0, 0.0, 3.0),
    dimensions_3d=(6.0, 3.0, 6.0), confidence=0.99,
    timestamp=0.0, frame_id='pi_frame',
)
graph = SceneGraph()
SceneGraphUpdater(graph).update([room, cube])

memory_path = Path('/tmp/agentic_memory_nav_workshop_v3.sqlite3')
if memory_path.exists():
    memory_path.unlink()
memory = SQLiteMemory(memory_path)
knowledge = KnowledgeMemory(memory)
facts_created = knowledge.materialize(graph)

print('graph nodes:', [(node.label, node.node_type.value) for node in graph.nodes()])
print('graph edges:', [(edge.relation, round(edge.confidence, 2)) for edge in graph.edges()])
print('knowledge facts created:', facts_created)
print('memory graph query:', [item.content for item in knowledge.retrieve_subgraph('cube kitchen')])

## 5. Memory Graph -> Native Reasoning -> Navigation

这一步是 agent 的关键：

1. parser 从任务文本提取 object 和 room；
2. reasoner 在 graph/memory graph 中寻找 object node；
3. 它验证 `inside(object, room)` relation；
4. planner 根据 reasoning evidence 产生高层 `NAVIGATE` waypoint；
5. 若目标不存在，planner 产生 `EXPLORE`，而不是编造目标位置。

当前 MVP parser 稳定支持 `Find cube in the kitchen`。颜色等细粒度信息已保存在 graph node attributes，可由更强的结构化 parser 在未来利用。

In [ ]:
from agentic_memory_nav.planning.rule_based_fallback import RuleBasedPlanner
from agentic_memory_nav.planning.task_parser import RuleBasedTaskParser
from agentic_memory_nav.reasoning.native_reasoner import NativeReasoner

task = RuleBasedTaskParser().parse('Find cube in the kitchen')
reasoning = NativeReasoner(knowledge).resolve(graph, task.parsed_goal)
plan = RuleBasedPlanner(approach_distance=0.6).plan(
    task=task, robot_pose=Pose3D(), graph=graph, memory=memory,
    replan_reason='new stable RGB-only submap',
)

print('parsed goal:', task.parsed_goal)
print('reasoning target:', reasoning.target_id)
print('reasoning evidence:', reasoning.evidence_ids)
print('navigation action:', plan.action.action_type.value)
print('target node:', plan.action.target)
print('waypoint:', plan.action.waypoint)
print('confidence:', round(plan.confidence, 3))
print('information gaps:', plan.information_gaps)

assert plan.action.action_type.value == 'navigate'
assert plan.action.target == reasoning.target_id
assert plan.action.waypoint is not None

## 6. 实时推理：MemoryAgent + NavigationAgent 逐帧驱动 Isaac Sim

真实 agent 是两个职责分离但每帧串联执行的 agent：

```text
Isaac Sim 实时渲染一帧 RGB + depth + camera/robot pose (真实数据，非离线回放)
-> MemoryAgent.ingest_frame(frame)  # 提取 + 构建 memory：mapper -> perception -> Pi -> Scene Graph -> Knowledge Memory
-> NavigationAgent.decide(pose, graph, memory)  # 只读最新 memory graph，输出 NAVIGATE / EXPLORE / VERIFY
-> Isaac Sim 执行该 action（send_waypoint），产生下一帧
```

`MemoryAgent` 和 `NavigationAgent` 是 `RealtimeAgent` 内部组合的两个子 agent（`agentic_memory_nav.agent.memory_agent` / `agentic_memory_nav.agent.navigation_agent`），职责与你在真实闭环里期望的完全一致：一个只负责 memory 提取与构建，一个只负责导航指令。

本 notebook 运行在项目的 Python 3.12 kernel 里，**不能**启动 Isaac Sim（Isaac Sim 需要它自带的 Python，通过 `~/isaacsim/python.sh` 启动）。所以这里做两件事来验证与 Isaac Sim 的匹配度：

1. 加载 `outputs/isaacsim_semantic_smoke` 下真实录制的 RGB/depth/内参/camera-to-world，逐帧喂给 `MemoryAgent` + `NavigationAgent`，并断言加载的 pose 与录制文件逐字节一致（不是编造数据）。
2. 展示真正的实时闭环脚本 `scripts/run_isaacsim_realtime_agent.py`：它在真实运行中的 Isaac Sim 里，每一步都调用 `executor.get_observation()` 拿到真实新帧，再驱动同样的 `MemoryAgent`/`NavigationAgent`，再把 action 通过 `executor.send_waypoint()` 写回 Isaac Sim，闭环持续进行。

In [ ]:
import json

from agentic_memory_nav.agent.memory_agent import MemoryAgent
from agentic_memory_nav.agent.navigation_agent import NavigationAgent

isaacsim_root = project_root / "outputs" / "isaacsim_semantic_smoke"
assert isaacsim_root.exists(), f"real Isaac Sim recording not found: {isaacsim_root}"

intrinsics_matrix = np.asarray(json.loads((isaacsim_root / "intrinsics.json").read_text()), dtype=np.float32)
camera_to_world = json.loads((isaacsim_root / "camera_to_world.json").read_text())
manifest = json.loads((isaacsim_root / "manifest.json").read_text())
print("real Isaac Sim manifest:", {k: v for k, v in manifest.items() if k != "lidar_scan_counts"})


def load_real_isaacsim_frame(index: int) -> FrameObservation:
    from PIL import Image

    depth_name = f"depth_{index:06d}.npy"
    rgb = np.asarray(Image.open(isaacsim_root / f"rgb/frame_{index:06d}.png").convert("RGB"), dtype=np.uint8)
    depth = np.load(isaacsim_root / "depth" / depth_name).astype(np.float32)
    c2w = np.asarray(camera_to_world[depth_name], dtype=np.float32)
    pose = Pose3D(position=tuple(float(value) for value in c2w[:3, 3]))
    # Verify this notebook is reading Isaac Sim's own recorded pose, not a synthetic one.
    assert np.allclose(pose.position, c2w[:3, 3], atol=0.0), "pose must match recorded camera_to_world exactly"
    return FrameObservation(
        frame_id=f"isaacsim_frame_{index:06d}",
        timestamp=float(index),
        rgb=rgb,
        depth=depth,
        camera_intrinsics=CameraIntrinsics(
            float(intrinsics_matrix[0, 0]), float(intrinsics_matrix[1, 1]),
            float(intrinsics_matrix[0, 2]), float(intrinsics_matrix[1, 2]),
            rgb.shape[1], rgb.shape[0],
        ),
        camera_pose=pose,
        robot_pose=pose,
        source="isaacsim",
    )


realtime_root = Path("/tmp/agentic_memory_nav_realtime_workshop")
memory_agent = MemoryAgent(realtime_root)
navigation_agent = NavigationAgent("Find the red cup in the kitchen")
try:
    for index in range(3):
        frame = load_real_isaacsim_frame(index)
        snapshot = memory_agent.ingest_frame(frame)
        plan = navigation_agent.decide(
            frame.robot_pose, memory_agent.graph, memory_agent.memory,
            replan_reason=f"new_rgb_frame:{frame.frame_id}",
        )
        print(
            f"frame {index} | isaacsim pose={frame.camera_pose.position} "
            f"action={plan.action.action_type.value} graph_nodes={snapshot.graph_nodes} "
            f"knowledge_facts={snapshot.knowledge_facts_created}"
        )
finally:
    memory_agent.close()

### 真正的实时闭环：Isaac Sim 内逐帧执行

上面的 cell 只能验证 `MemoryAgent`/`NavigationAgent` 与录制数据的匹配度；无法在这个 kernel 里启动真正运行中的 Isaac Sim。真实的实时闭环用同一对 agent，通过 `scripts/run_isaacsim_realtime_agent.py` 驱动：

```bash
~/isaacsim/python.sh scripts/run_isaacsim_realtime_agent.py \
    --config configs/isaacsim_realtime_agent.yaml
```

每一步都是：`IsaacSimExecutor.get_observation()`（真实渲染的 RGB + depth + pose）-> `MemoryAgent.ingest_frame()` -> `NavigationAgent.decide()` -> `IsaacSimExecutor.send_waypoint()`，然后用这一步执行后的新状态渲染下一帧，循环持续到成功、碰撞或达到步数上限。`configs/isaacsim_realtime_agent.yaml` 里的 `perception.backend` 可以从 `mock` 切换为 `vlm`，接入 `10.6.32.16:8000/v1` 上的 `Inferact/Qwen3.8-27B-NVFP4`。

### 没有 memory graph 证据：Explore

这是 agent 的认识论约束：目标未被观察到时，先安全探索并收集新的 RGB、LingBot local submap 和 VLM 语义证据，而不是编造导航位置。

In [ ]:
empty_graph = SceneGraph()
empty_path = Path('/tmp/agentic_memory_nav_workshop_v3_empty.sqlite3')
if empty_path.exists():
    empty_path.unlink()
empty_memory = SQLiteMemory(empty_path)
explore_plan = RuleBasedPlanner().plan(
    task=task, robot_pose=Pose3D(), graph=empty_graph, memory=empty_memory
)

print('navigation action:', explore_plan.action.action_type.value)
print('exploration waypoint:', explore_plan.action.waypoint)
print('information gaps:', explore_plan.information_gaps)
assert explore_plan.action.action_type.value == 'explore'
assert explore_plan.replan_required

empty_memory.close()
memory.close()

## 7. Runtime 与离线评测的边界

运行时：

```text
new RGB frame -> LingBot depth/c2w -> stable local submap -> spatial memory
new RGB frame -> VLM objects/triples -> scene graph + knowledge memory
updated memory graph -> native reasoning -> NAVIGATE / EXPLORE / VERIFY
```

每一帧的 action 都来自最新 graph/memory state，而不是固定脚本。外部控制器执行当前 action 后，下一帧又会触发一次新的推理与重规划。

GT depth、GT c2w 与 LiDAR 仅用于离线评测。stable RGB-only submap 会携带 overlap residual、confidence 与 provenance 写入 memory；低置信度、冲突关系或 room evidence 缺失时，agent 应重新观察或执行 `VERIFY`。